# 📚 Liane's Library — CRUD Operations

Before writing any sort of application, CRUD functions need to be developed and tested. This notebook provides a space for that.

> **C**reate · **R**ead · **U**pdate · **D**elete · **V**alidate

---

## 0. Setup — imports and connection 🔌

In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from datetime import date

In [2]:
# Load password from .env file
# Make sure your .env file contains: MYSQL_PASSWORD=your_password_here
load_dotenv(override=True)

schema   = "lianes_library"
host     = "127.0.0.1"
user     = "root"
password = os.getenv("MYSQL_PASSWORD")
port     = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
engine = create_engine(connection_string)

print("Connected to lianes_library!")

Connected to lianes_library!


---
# CREATE ➕
Functions to add new records into the database.

## Define

In [3]:
def create_book(isbn, book_name, author, genre=None,
                published_year=None, total_pages=None,
                book_condition='good', mood_tags=None):
    """
    Adds a new book to Liane's collection.
    Skips safely if ISBN already exists.
    book_condition must be one of: mint, good, worn, damaged
    mood_tags example: 'cosy,dark,inspiring'
    """
    existing = pd.read_sql(
        f"SELECT isbn FROM books WHERE isbn = '{isbn}'",
        con=connection_string
    )
    if not existing.empty:
        return f"⚠️ ISBN '{isbn}' already exists — skipping."

    df = pd.DataFrame([{
        "isbn"          : isbn,
        "book_name"     : book_name,
        "author"        : author,
        "genre"         : genre,
        "published_year": published_year,
        "total_pages"   : total_pages,
        "book_condition": book_condition,
        "mood_tags"     : mood_tags,
        "is_available"  : True
    }])
    df.to_sql("books", if_exists="append", con=connection_string, index=False)
    return f"📖 '{book_name}' by {author} added to the library!"

In [4]:
def create_friend(friend_name, phone_number=None, email=None,
                  max_loans=3, trust_score=100,
                  preferred_genres=None, notes=None):
    """
    Adds a new friend/borrower to the system.
    Skips safely if email already exists.
    trust_score starts at 100 and goes down for bad behaviour.
    """
    if email:
        existing = pd.read_sql(
            f"SELECT email FROM friends WHERE email = '{email}'",
            con=connection_string
        )
        if not existing.empty:
            return f"⚠️ '{friend_name}' with email '{email}' already exists — skipping."

    df = pd.DataFrame([{
        "friend_name"     : friend_name,
        "phone_number"    : phone_number,
        "email"           : email,
        "max_loans"       : max_loans,
        "trust_score"     : trust_score,
        "preferred_genres": preferred_genres,
        "notes"           : notes
    }])
    df.to_sql("friends", if_exists="append", con=connection_string, index=False)
    return f"👤 '{friend_name}' added as a borrower!"

In [5]:
def create_loan(friend, book, loan_date=None, due_days=14, notes=None):
    """
    Records a new loan.
    Skips safely if this friend already has this book on loan.
    due_days = how many days until the book is due back (default 14).
    renewal_date is set automatically to due_date + 7 days.
    tracker_id is generated automatically by the database trigger.
    """
    if loan_date is None:
        loan_date = date.today()

    # Fixed: use 'D' instead of deprecated 'd'
    due_date     = pd.Timestamp(loan_date) + pd.Timedelta(due_days, "D")
    renewal_date = due_date + pd.Timedelta(7, "D")

    # Skip if active loan already exists for same friend + book
    existing = pd.read_sql(f"""
        SELECT loan_id FROM loans
        WHERE isbn = '{book['isbn']}'
        AND friend_id = {friend['friend_id']}
        AND return_date IS NULL
    """, con=connection_string)

    if not existing.empty:
        return f"⚠️ '{friend['friend_name']}' already has '{book['book_name']}' on loan — skipping."

    df = pd.DataFrame([{
        "isbn"        : book["isbn"],
        "friend_id"   : friend["friend_id"],
        "loan_date"   : loan_date,
        "due_date"    : due_date.date(),
        "renewal_date": renewal_date.date(),
        "notes"       : notes
    }])
    df.to_sql("loans", if_exists="append", con=connection_string, index=False)
    return f"📋 '{friend['friend_name']}' borrowed '{book['book_name']}'. Due: {due_date.date()}"

In [6]:
def create_reading_session(loan_id, pages_read, mood_rating=None,
                            session_notes=None, session_date=None):
    """
    Logs a reading session for a loan.
    mood_rating: 1 (terrible) to 5 (amazing)
    This is unique to Liane's system — tracks reading progress!
    """
    if session_date is None:
        session_date = date.today()

    df = pd.DataFrame([{
        "loan_id"      : loan_id,
        "session_date" : session_date,
        "pages_read"   : pages_read,
        "mood_rating"  : mood_rating,
        "session_notes": session_notes
    }])
    df.to_sql("reading_sessions", if_exists="append", con=connection_string, index=False)
    return f"📖 Reading session logged! {pages_read} pages read for loan {loan_id}."

In [7]:
def create_wishlist_request(friend, book_title, author=None):
    """
    Adds a book request to the wishlist.
    Skips if the same friend already requested the same title.
    """
    existing = pd.read_sql(f"""
        SELECT wishlist_id FROM wishlist
        WHERE friend_id = {friend['friend_id']}
        AND book_title = '{book_title}'
    """, con=connection_string)

    if not existing.empty:
        return f"⚠️ '{friend['friend_name']}' already requested '{book_title}' — skipping."

    df = pd.DataFrame([{
        "friend_id" : friend["friend_id"],
        "book_title": book_title,
        "author"    : author,
        "fulfilled" : False
    }])
    df.to_sql("wishlist", if_exists="append", con=connection_string, index=False)
    return f"🎁 '{friend['friend_name']}' requested '{book_title}'. Added to wishlist!"

In [8]:
def create_review(loan_id, rating, review_text=None):
    """
    Adds a review after a book is returned.
    rating: 1 to 5 stars.
    Skips if a review already exists for this loan.
    """
    existing = pd.read_sql(
        f"SELECT review_id FROM reviews WHERE loan_id = {loan_id}",
        con=connection_string
    )
    if not existing.empty:
        return f"⚠️ A review already exists for loan {loan_id} — skipping."

    df = pd.DataFrame([{
        "loan_id"    : loan_id,
        "rating"     : rating,
        "review_text": review_text,
        "reviewed_on": date.today()
    }])
    df.to_sql("reviews", if_exists="append", con=connection_string, index=False)
    stars = "⭐" * rating
    return f"{stars} Review submitted for loan {loan_id}!"

## Test CREATE

In [9]:
# Add new books — skips automatically if already exists
print(create_book("9780316769174", "The Catcher in the Rye",     "J.D. Salinger",   "Fiction", 1951, 277,  "good", "dark,emotional"))
print(create_book("9780143127741", "The Wind-Up Bird Chronicle",  "Haruki Murakami", "Fiction", 1994, 607,  "mint", "mysterious,cosy"))
print(create_book("9781501173219", "It",                          "Stephen King",    "Horror",  1986, 1138, "good", "dark,intense"))

pd.read_sql("SELECT isbn, book_name, author, genre FROM books", con=connection_string)

⚠️ ISBN '9780316769174' already exists — skipping.
⚠️ ISBN '9780143127741' already exists — skipping.
⚠️ ISBN '9781501173219' already exists — skipping.


,isbn,book_name,author,genre
0,9780062316097,Sapiens,Yuval Noah Harari,Non-Fiction
1,9780141036144,To Kill a Mockingbird,Harper Lee,Fiction
2,9780143127741,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction
3,9780241984536,The Alchemist,Paulo Coelho,Fiction
4,9780316769174,The Catcher in the Rye,J.D. Salinger,Fiction
5,9780385490818,The Handmaids Tale,Margaret Atwood,Dystopian
6,9780525559474,The Subtle Art of Not Giving a F,Mark Manson,Self-Help
7,9780593311295,Atomic Habits,James Clear,Self-Help
8,9780679720201,Crime and Punishment,Fyodor Dostoevsky,Classic
9,9780743273565,The Great Gatsby,F. Scott Fitzgerald,Fiction


In [10]:
# Add new friends — skips automatically if email already exists
print(create_friend("Ashritha",    "07811111111", "ashritha@email.com", 3, 100, "Romance,Fiction",   "Very trustworthy"))
print(create_friend("Noah Bennett","07822222222", "noah@email.com",     2, 90,  "Dystopian,Classic", "Sometimes slow to return"))
print(create_friend("Zara Ahmed",  "07833333333", "zara@email.com",     3, 100, "Self-Help,Fiction",  "Always on time"))

pd.read_sql("SELECT friend_id, friend_name, email, trust_score FROM friends", con=connection_string)

⚠️ 'Ashritha' with email 'ashritha@email.com' already exists — skipping.
⚠️ 'Noah Bennett' with email 'noah@email.com' already exists — skipping.
⚠️ 'Zara Ahmed' with email 'zara@email.com' already exists — skipping.


,friend_id,friend_name,email,trust_score
0,1,Emma Watson,emma@email.com,100
1,2,James Brown,james@email.com,55
2,3,Sophie Turner,sophie@email.com,90
3,4,Liam Smith,liam@email.com,60
4,5,Olivia Jones,olivia@email.com,100
5,6,Liane herself,liane@library.com,100
6,16,Ashritha,ashritha@email.com,100
7,17,Noah Bennett,noah@email.com,90
8,18,Zara Ahmed,zara@email.com,100


In [11]:
# Create loans — skips automatically if active loan already exists
friend = pd.read_sql("SELECT * FROM friends WHERE friend_name = 'Ashritha'",     con=connection_string).iloc[0]
book   = pd.read_sql("SELECT * FROM books   WHERE isbn = '9780241984536'",        con=connection_string).iloc[0]
print(create_loan(friend, book, due_days=14, notes="Requested this one specifically"))

friend = pd.read_sql("SELECT * FROM friends WHERE friend_name = 'Noah Bennett'", con=connection_string).iloc[0]
book   = pd.read_sql("SELECT * FROM books   WHERE isbn = '9780143127741'",        con=connection_string).iloc[0]
print(create_loan(friend, book, due_days=21, notes="Loves dystopian fiction"))

friend = pd.read_sql("SELECT * FROM friends WHERE friend_name = 'Zara Ahmed'",   con=connection_string).iloc[0]
book   = pd.read_sql("SELECT * FROM books   WHERE isbn = '9781501173219'",        con=connection_string).iloc[0]
print(create_loan(friend, book))

pd.read_sql("SELECT loan_id, tracker_id, isbn, friend_id, loan_date, due_date FROM loans", con=connection_string)

⚠️ 'Ashritha' already has 'The Alchemist' on loan — skipping.
📋 'Noah Bennett' borrowed 'The Wind-Up Bird Chronicle'. Due: 2026-10-05
📋 'Zara Ahmed' borrowed 'It'. Due: 2026-09-28


,loan_id,tracker_id,isbn,friend_id,loan_date,due_date
0,1,LIB-000001,9780141036144,1,2024-08-01,2024-08-15
1,2,LIB-000002,9780743273565,2,2024-08-05,2024-08-19
2,3,LIB-000003,9780062316097,3,2024-08-10,2024-08-24
3,4,LIB-000004,9780747532743,4,2024-08-12,2024-08-26
4,5,LIB-000005,9781501156700,5,2024-08-15,2024-08-29
5,6,LIB-000006,9780593311295,1,2024-08-20,2024-09-03
6,7,LIB-000007,9780385490818,2,2024-08-22,2024-09-05
7,8,LIB-000008,9780679720201,3,2024-08-25,2024-09-08
8,9,LIB-000009,9781250301697,4,2024-09-01,2024-09-15
9,10,LIB-000010,9780525559474,5,2024-09-05,2024-09-19


In [12]:
# Log reading sessions for the 3 most recent loans
recent_loans = pd.read_sql("SELECT * FROM loans ORDER BY loan_id DESC LIMIT 3", con=connection_string)

print(create_reading_session(recent_loans.iloc[0]["loan_id"], pages_read=80,  mood_rating=5, session_notes="Absolutely gripping!"))
print(create_reading_session(recent_loans.iloc[1]["loan_id"], pages_read=55,  mood_rating=4, session_notes="Heavy themes but brilliant"))
print(create_reading_session(recent_loans.iloc[2]["loan_id"], pages_read=100, mood_rating=5, session_notes="Could not put it down!"))

pd.read_sql("reading_sessions", con=connection_string)

📖 Reading session logged! 80 pages read for loan 15.
📖 Reading session logged! 55 pages read for loan 14.
📖 Reading session logged! 100 pages read for loan 11.


,session_id,loan_id,session_date,pages_read,mood_rating,session_notes
0,1,1,2024-08-02,50,5,Amazing start
1,2,1,2024-08-05,80,5,Could not put it down
2,3,2,2024-08-06,30,3,Slow start
3,4,3,2024-08-11,60,4,Really enjoying it
4,5,4,2024-08-13,40,3,Getting into it slowly
5,6,5,2024-08-16,100,5,Finished in one sitting
6,7,6,2024-08-21,45,4,Very practical book
7,8,7,2024-08-23,55,4,Dark but gripping
8,9,8,2024-08-26,70,5,Intense reading
9,10,9,2024-09-02,90,5,Beautiful story


In [13]:
# Add wishlist requests — skips if already requested
friend = pd.read_sql("SELECT * FROM friends WHERE friend_name = 'Zara Ahmed'",   con=connection_string).iloc[0]
print(create_wishlist_request(friend, "The Power of Now", "Eckhart Tolle"))

friend = pd.read_sql("SELECT * FROM friends WHERE friend_name = 'Noah Bennett'", con=connection_string).iloc[0]
print(create_wishlist_request(friend, "1984", "George Orwell"))

pd.read_sql("wishlist", con=connection_string)

🎁 'Zara Ahmed' requested 'The Power of Now'. Added to wishlist!
🎁 'Noah Bennett' requested '1984'. Added to wishlist!


,wishlist_id,friend_id,book_title,author,requested_on,fulfilled
0,1,1,The Alchemist,Paulo Coelho,2024-08-10,1
1,2,2,Thinking Fast and Slow,Daniel Kahneman,2024-08-12,0
2,3,3,Pride and Prejudice,Jane Austen,2024-08-15,0
3,4,4,1984,George Orwell,2024-08-18,0
4,5,5,The Power of Now,Eckhart Tolle,2024-08-20,0
5,6,18,The Power of Now,Eckhart Tolle,2026-09-14,0
6,7,17,1984,George Orwell,2026-09-14,0


---
# READ 🔍
Functions to retrieve and display data beautifully.

## Define

In [ ]:
def prettify_df(df):
    """Makes column names clean and readable for display."""
    df = df.copy()
    df.columns = [
        c.upper() if c == "isbn" else c.replace("_", " ").capitalize()
        for c in df.columns
    ]
    return df.fillna("")

In [ ]:
def read_books(available_only=False, genre=None, mood=None):
    """
    Reads all books.
    available_only=True  → only books currently on the shelf.
    genre='Fiction'      → filter by genre.
    mood='cosy'          → filter by mood tag.
    """
    books = pd.read_sql("books", con=connection_string)
    if available_only:
        books = books[books["is_available"] == 1]
    if genre:
        books = books[books["genre"].str.lower() == genre.lower()]
    if mood:
        books = books[books["mood_tags"].str.contains(mood, na=False)]
    return books

def display_books(books):
    """Returns a clean sorted view of books."""
    return books.pipe(prettify_df).sort_values(by="Book name")

In [ ]:
def read_friends(min_trust=None):
    """
    Reads all friends.
    min_trust=80 → only friends with trust score 80 or above.
    """
    friends = pd.read_sql("friends", con=connection_string)
    if min_trust:
        friends = friends[friends["trust_score"] >= min_trust]
    return friends

def display_friends(friends):
    """Returns a clean sorted view of friends by trust score."""
    return friends.pipe(prettify_df).sort_values(by="Trust score", ascending=False)

In [ ]:
def read_loans(active_only=False):
    """Reads all loans. active_only=True → only unreturned loans."""
    loans = pd.read_sql("loans", con=connection_string)
    if active_only:
        loans = loans[loans["return_date"].isna()]
    return loans

def display_loans():
    """Returns a joined view of loans with book and friend names."""
    return pd.read_sql("""
        SELECT
            l.tracker_id,
            f.friend_name,
            b.book_name,
            l.loan_date,
            l.due_date,
            l.renewal_date,
            CASE
                WHEN l.return_date IS NOT NULL                         THEN 'Returned'
                WHEN CURDATE() > COALESCE(l.renewal_date, l.due_date) THEN 'OVERDUE'
                WHEN l.renewal_date IS NOT NULL                       THEN 'Active - Renewed'
                ELSE 'Active'
            END AS status,
            l.fine_amount,
            l.fine_status
        FROM loans l
        JOIN books   b ON l.isbn      = b.isbn
        JOIN friends f ON l.friend_id = f.friend_id
        ORDER BY l.due_date ASC;
    """, con=connection_string)

In [ ]:
def check_tracker(tracker_id):
    """Look up a single loan using its tracker ID like LIB-000001."""
    return pd.read_sql(f"""
        SELECT
            l.tracker_id,
            f.friend_name,
            b.book_name,
            l.loan_date,
            l.due_date,
            l.renewal_date,
            l.return_date,
            l.fine_amount,
            l.fine_status
        FROM loans l
        JOIN books   b ON l.isbn      = b.isbn
        JOIN friends f ON l.friend_id = f.friend_id
        WHERE l.tracker_id = '{tracker_id}';
    """, con=connection_string)

In [ ]:
def read_overdue():
    """Returns all overdue loans with days overdue and fine calculated."""
    return pd.read_sql("""
        SELECT
            l.tracker_id,
            f.friend_name,
            f.phone_number,
            b.book_name,
            COALESCE(l.renewal_date, l.due_date)                       AS deadline,
            DATEDIFF(CURDATE(), COALESCE(l.renewal_date, l.due_date))  AS days_overdue,
            DATEDIFF(CURDATE(), COALESCE(l.renewal_date, l.due_date)) * 0.50 AS fine_due
        FROM loans l
        JOIN books   b ON l.isbn      = b.isbn
        JOIN friends f ON l.friend_id = f.friend_id
        WHERE l.return_date IS NULL
        AND CURDATE() > COALESCE(l.renewal_date, l.due_date)
        ORDER BY days_overdue DESC;
    """, con=connection_string)

In [ ]:
def read_reading_progress():
    """Shows reading progress per loan as a percentage."""
    return pd.read_sql("""
        SELECT
            f.friend_name,
            b.book_name,
            b.total_pages,
            SUM(rs.pages_read) AS pages_read_so_far,
            ROUND(SUM(rs.pages_read) * 100.0 / b.total_pages, 1) AS percent_complete,
            ROUND(AVG(rs.mood_rating), 1) AS avg_mood
        FROM reading_sessions rs
        JOIN loans   l ON rs.loan_id  = l.loan_id
        JOIN books   b ON l.isbn      = b.isbn
        JOIN friends f ON l.friend_id = f.friend_id
        GROUP BY f.friend_name, b.book_name, b.total_pages
        ORDER BY percent_complete DESC;
    """, con=connection_string)

In [ ]:
def library_summary():
    """One-line dashboard of the whole library."""
    return pd.read_sql("""
        SELECT
            (SELECT COUNT(*) FROM books)                                AS total_books,
            (SELECT COUNT(*) FROM books WHERE is_available = 1)        AS available,
            (SELECT COUNT(*) FROM friends)                             AS total_friends,
            (SELECT COUNT(*) FROM loans WHERE return_date IS NULL)     AS active_loans,
            (SELECT COUNT(*) FROM loans
             WHERE return_date IS NULL
             AND CURDATE() > COALESCE(renewal_date, due_date))         AS overdue_loans,
            (SELECT COALESCE(SUM(fine_amount), 0)
             FROM loans WHERE fine_status = 'unpaid')                  AS unpaid_fines,
            (SELECT COUNT(*) FROM wishlist WHERE fulfilled = 0)        AS wishlist_pending;
    """, con=connection_string)

## Test READ

In [ ]:
# All books
display_books(read_books())

In [ ]:
# Only available books
display_books(read_books(available_only=True))

In [ ]:
# Filter by genre
display_books(read_books(genre="Fiction"))

In [ ]:
# Filter by mood tag — great for recommendations!
display_books(read_books(mood="cosy"))

In [ ]:
# All friends sorted by trust score
display_friends(read_friends())

In [ ]:
# Only trustworthy friends (score 80+)
display_friends(read_friends(min_trust=80))

In [ ]:
# Full loans overview
display_loans()

In [ ]:
# Check one specific loan by tracker ID
check_tracker('LIB-000001')

In [ ]:
# See all overdue loans with fines
read_overdue()

In [ ]:
# Reading progress per borrower
read_reading_progress()

In [ ]:
# Full library summary dashboard
library_summary()

---
# UPDATE ✏️
Functions to modify existing records.

## Define

In [ ]:
def format_field(field):
    """Makes field names human readable for messages."""
    special = {
        "isbn": "ISBN", "trust_score": "Trust score",
        "fine_status": "Fine status", "is_available": "Availability"
    }
    return special.get(field, field.replace("_", " ").capitalize())

In [ ]:
def update_friend(friend, field, new_value):
    """
    Updates any field on a friend record.
    e.g. update_friend(friend, 'trust_score', 80)
    """
    query = f"""
        UPDATE friends
        SET {field} = '{new_value}'
        WHERE friend_id = {friend['friend_id']};
    """
    with engine.begin() as conn:
        conn.execute(text(query))
    return f"✅ {format_field(field)} updated for '{friend['friend_name']}' → '{new_value}'"

In [ ]:
def update_book(book, field, new_value):
    """
    Updates any field on a book record.
    e.g. update_book(book, 'book_condition', 'worn')
    """
    query = f"""
        UPDATE books
        SET {field} = '{new_value}'
        WHERE isbn = '{book['isbn']}';
    """
    with engine.begin() as conn:
        conn.execute(text(query))
    return f"✅ {format_field(field)} updated for '{book['book_name']}' → '{new_value}'"

In [ ]:
def return_book(tracker_id, return_condition='good'):
    """
    Marks a book as returned.
    The calculate_fine trigger fires automatically and sets the fine.
    return_condition: mint, good, worn, damaged
    """
    query = f"""
        UPDATE loans
        SET return_date      = CURDATE(),
            return_condition = '{return_condition}'
        WHERE tracker_id = '{tracker_id}';
    """
    with engine.begin() as conn:
        conn.execute(text(query))
    return f"📬 Loan {tracker_id} returned in '{return_condition}' condition. Fine calculated automatically."

In [ ]:
def renew_loan(tracker_id, extra_days=7):
    """
    Extends a loan by adding extra days to the renewal date.
    Default is 7 extra days.
    """
    query = f"""
        UPDATE loans
        SET renewal_date = DATE_ADD(COALESCE(renewal_date, due_date), INTERVAL {extra_days} DAY)
        WHERE tracker_id = '{tracker_id}';
    """
    with engine.begin() as conn:
        conn.execute(text(query))
    return f"🔄 Loan {tracker_id} renewed by {extra_days} days."

In [ ]:
def pay_fine(tracker_id):
    """Marks a fine as paid for a given tracker ID."""
    query = f"""
        UPDATE loans
        SET fine_status = 'paid'
        WHERE tracker_id = '{tracker_id}';
    """
    with engine.begin() as conn:
        conn.execute(text(query))
    return f"💰 Fine marked as PAID for loan {tracker_id}."

In [ ]:
def lower_trust_score(friend, points=10, reason=None):
    """
    Lowers a friend's trust score by a given number of points.
    Default is -10 points. Will not go below 0.
    """
    query = f"""
        UPDATE friends
        SET trust_score = GREATEST(0, trust_score - {points})
        WHERE friend_id = {friend['friend_id']};
    """
    with engine.begin() as conn:
        conn.execute(text(query))
    msg = f"⚠️ Trust score for '{friend['friend_name']}' reduced by {points} points."
    if reason:
        msg += f" Reason: {reason}"
    return msg

In [ ]:
def fulfill_wishlist(wishlist_id):
    """Marks a wishlist request as fulfilled when Liane gets the book."""
    query = f"""
        UPDATE wishlist
        SET fulfilled = TRUE
        WHERE wishlist_id = {wishlist_id};
    """
    with engine.begin() as conn:
        conn.execute(text(query))
    return f"🎁 Wishlist item {wishlist_id} marked as fulfilled!"

## Test UPDATE

In [ ]:
# Update a friend's details
friend = pd.read_sql("SELECT * FROM friends WHERE friend_name = 'Noah Bennett'", con=connection_string).iloc[0]

print(update_friend(friend, 'max_loans', 3))
print(update_friend(friend, 'notes', 'Updated - now very reliable'))

pd.read_sql("SELECT * FROM friends WHERE friend_name = 'Noah Bennett'", con=connection_string)

In [ ]:
# Update a book's condition and mood tags
book = pd.read_sql("SELECT * FROM books WHERE isbn = '9780241984536'", con=connection_string).iloc[0]

print(update_book(book, 'book_condition', 'worn'))
print(update_book(book, 'mood_tags', 'inspiring,magical,emotional'))

pd.read_sql("SELECT * FROM books WHERE isbn = '9780241984536'", con=connection_string)

In [ ]:
# Renew a loan
print(renew_loan('LIB-000001', extra_days=7))
check_tracker('LIB-000001')

In [ ]:
# Return a book — fine calculated automatically by trigger
print(return_book('LIB-000002', return_condition='worn'))
check_tracker('LIB-000002')

In [ ]:
# Pay a fine
print(pay_fine('LIB-000002'))
check_tracker('LIB-000002')

In [ ]:
# Lower trust score for a friend who returned a book damaged
friend = pd.read_sql("SELECT * FROM friends WHERE friend_name = 'Noah Bennett'", con=connection_string).iloc[0]
print(lower_trust_score(friend, points=15, reason="Returned book with water damage"))

pd.read_sql("SELECT friend_name, trust_score FROM friends ORDER BY trust_score DESC", con=connection_string)

In [ ]:
# Mark a wishlist item as fulfilled
wishlist = pd.read_sql("SELECT * FROM wishlist WHERE fulfilled = 0", con=connection_string)
if not wishlist.empty:
    print(fulfill_wishlist(wishlist.iloc[0]['wishlist_id']))
pd.read_sql("wishlist", con=connection_string)

---
# DELETE 🗑️
Functions to remove records safely.

## Define

In [ ]:
def delete_friend(friend):
    """
    Removes a friend from the system.
    Warning: will fail if the friend has active loans.
    """
    # Safety check — block if friend has active loans
    active = pd.read_sql(
        f"SELECT loan_id FROM loans WHERE friend_id = {friend['friend_id']} AND return_date IS NULL",
        con=connection_string
    )
    if not active.empty:
        return f"⚠️ '{friend['friend_name']}' still has {len(active)} active loan(s) — cannot delete."

    query = f"DELETE FROM friends WHERE friend_id = {friend['friend_id']};"
    with engine.begin() as conn:
        conn.execute(text(query))
    return f"🗑️ '{friend['friend_name']}' removed from friends."

In [ ]:
def delete_book(book):
    """
    Removes a book from the library.
    Warning: will fail if the book is currently on loan.
    """
    # Safety check — block if book is on active loan
    active = pd.read_sql(
        f"SELECT loan_id FROM loans WHERE isbn = '{book['isbn']}' AND return_date IS NULL",
        con=connection_string
    )
    if not active.empty:
        return f"⚠️ '{book['book_name']}' is currently on loan — cannot delete."

    query = f"DELETE FROM books WHERE isbn = '{book['isbn']}';"
    with engine.begin() as conn:
        conn.execute(text(query))
    return f"🗑️ '{book['book_name']}' removed from the library."

In [ ]:
def delete_loan(tracker_id):
    """
    Removes a loan record by tracker ID.
    Also deletes linked reading sessions and reviews first.
    """
    loan = pd.read_sql(
        f"SELECT * FROM loans WHERE tracker_id = '{tracker_id}'",
        con=connection_string
    )
    if loan.empty:
        return f"⚠️ No loan found with tracker ID '{tracker_id}'."

    loan_id = int(loan.iloc[0]["loan_id"])
    with engine.begin() as conn:
        conn.execute(text(f"DELETE FROM reading_sessions WHERE loan_id = {loan_id};"))
        conn.execute(text(f"DELETE FROM reviews          WHERE loan_id = {loan_id};"))
        conn.execute(text(f"DELETE FROM loans            WHERE loan_id = {loan_id};"))

    return f"🗑️ Loan {tracker_id} and its linked sessions/reviews deleted."

In [ ]:
def delete_wishlist(wishlist_id):
    """Removes a wishlist request."""
    query = f"DELETE FROM wishlist WHERE wishlist_id = {wishlist_id};"
    with engine.begin() as conn:
        conn.execute(text(query))
    return f"🗑️ Wishlist item {wishlist_id} removed."

## Test DELETE

In [ ]:
# Check friends table before deleting
table_pre = pd.read_sql("friends", con=connection_string)
table_pre

In [ ]:
# Delete the last added friend — safely blocked if they have active loans
friend = pd.read_sql("SELECT * FROM friends ORDER BY friend_id DESC LIMIT 1", con=connection_string).iloc[0]
print(delete_friend(friend))

In [ ]:
# Confirm the row is gone — compare before and after
table_post = pd.read_sql("friends", con=connection_string)
dropped = pd.concat([table_pre, table_post]).drop_duplicates(keep=False)
print("Removed row:")
dropped

In [ ]:
# Delete a loan safely — also removes linked sessions and reviews
# Use the last loan in the table so we don't remove important original data
last_loan = pd.read_sql("SELECT tracker_id FROM loans ORDER BY loan_id DESC LIMIT 1", con=connection_string)
if not last_loan.empty:
    tracker = last_loan.iloc[0]['tracker_id']
    loans_pre = pd.read_sql("loans", con=connection_string)
    print(delete_loan(tracker))
    loans_post = pd.read_sql("loans", con=connection_string)
    dropped_loan = pd.concat([loans_pre, loans_post]).drop_duplicates(keep=False)
    print("\nRemoved loan:")
    print(dropped_loan[['loan_id','tracker_id','isbn','friend_id']].to_string(index=False))

---
# VALIDATE ✅
Validation functions that check data BEFORE it goes into the database.
These are called inside the Streamlit app to show warnings to the user.

## Define

In [ ]:
def validate_name(name):
    """Checks a name is not empty."""
    if not name or not str(name).strip():
        return "⚠️ Name cannot be empty."
    return ""

In [ ]:
def validate_isbn(isbn):
    """Checks ISBN is 10 or 13 digits and not already in the database."""
    isbn = str(isbn)
    if len(isbn) not in (10, 13):
        return "⚠️ ISBN must be 10 or 13 digits long."
    if not isbn.isnumeric():
        return "⚠️ ISBN must contain numbers only."
    existing = pd.read_sql("SELECT isbn FROM books", con=connection_string)["isbn"].values
    if isbn in existing:
        return "⚠️ This ISBN already exists in the library."
    return ""

In [ ]:
def validate_email(email):
    """Checks email has basic valid format and is not already registered."""
    if not email or "@" not in str(email) or "." not in str(email):
        return "⚠️ Please enter a valid email address."
    existing = pd.read_sql("SELECT email FROM friends", con=connection_string)["email"].values
    if email in existing:
        return "⚠️ This email is already registered."
    return ""

In [ ]:
def validate_loan_friend(friend):
    """
    Checks a friend is allowed to borrow another book.
    Blocks if they have reached their max_loans limit.
    Warns if trust score is below 50.
    """
    active_loans = pd.read_sql(
        f"SELECT loan_id FROM loans WHERE friend_id = {friend['friend_id']} AND return_date IS NULL",
        con=connection_string
    )
    if len(active_loans) >= friend["max_loans"]:
        return f"⚠️ '{friend['friend_name']}' has reached their max loan limit ({friend['max_loans']})."
    if friend["trust_score"] < 50:
        return f"⚠️ '{friend['friend_name']}' has a low trust score ({friend['trust_score']}). Lend with caution!"
    return ""

In [ ]:
def validate_loan_book(book):
    """Checks a book is available to be borrowed."""
    on_loan = pd.read_sql(
        f"SELECT loan_id FROM loans WHERE isbn = '{book['isbn']}' AND return_date IS NULL",
        con=connection_string
    )
    if not on_loan.empty:
        return f"⚠️ '{book['book_name']}' is already out on loan."
    if book["book_condition"] == "damaged":
        return f"⚠️ '{book['book_name']}' is marked as damaged. Should it be lent out?"
    return ""

In [ ]:
def validate_rating(rating):
    """Checks rating is a whole number between 1 and 5."""
    if not isinstance(rating, int) or rating < 1 or rating > 5:
        return "⚠️ Rating must be a whole number between 1 and 5."
    return ""

## Test VALIDATE

In [ ]:
# Name validation
print(validate_name("Liane"))   # ✅ should pass
print(validate_name(""))         # ⚠️ empty
print(validate_name("   "))      # ⚠️ only spaces
print(validate_name(None))       # ⚠️ None

In [ ]:
# ISBN validation
print(validate_isbn("9780141036144"))  # ⚠️ already in DB
print(validate_isbn("123"))             # ⚠️ too short
print(validate_isbn("97801410361ab"))   # ⚠️ not numeric
print(validate_isbn("9999999999999"))   # ✅ valid new ISBN

In [ ]:
# Email validation
print(validate_email("emma@email.com"))  # ⚠️ already registered
print(validate_email("notanemail"))       # ⚠️ invalid format
print(validate_email("brand.new@email.com"))  # ✅ valid

In [ ]:
# Loan friend validation
friend_ok  = pd.read_sql("SELECT * FROM friends WHERE friend_name = 'Emma Watson'", con=connection_string).iloc[0]
friend_bad = pd.read_sql("SELECT * FROM friends WHERE friend_name = 'Liam Smith'",  con=connection_string).iloc[0]

print(validate_loan_friend(friend_ok))   # ✅ should pass
print(validate_loan_friend(friend_bad))  # ⚠️ low trust score

In [ ]:
# Loan book validation
book_available = pd.read_sql("SELECT * FROM books WHERE isbn = '9780316769174'", con=connection_string).iloc[0]  # Catcher in the Rye
book_on_loan   = pd.read_sql("SELECT * FROM books WHERE isbn = '9780141036144'", con=connection_string).iloc[0]  # Mockingbird

print(validate_loan_book(book_available))  # ✅ should pass
print(validate_loan_book(book_on_loan))    # ⚠️ already on loan

In [ ]:
# Rating validation
print(validate_rating(5))    # ✅ should pass
print(validate_rating(0))    # ⚠️ too low
print(validate_rating(6))    # ⚠️ too high
print(validate_rating(3.5))  # ⚠️ not integer

---
# All functions are ready ✅

These CRUD functions can now be imported directly into the Streamlit app:

```python
from crud import (
    create_book, create_friend, create_loan,
    create_reading_session, create_wishlist_request, create_review,
    read_books, read_friends, display_loans, check_tracker,
    read_overdue, read_reading_progress, library_summary,
    update_friend, update_book, return_book, renew_loan,
    pay_fine, lower_trust_score, fulfill_wishlist,
    delete_friend, delete_book, delete_loan, delete_wishlist,
    validate_name, validate_isbn, validate_email,
    validate_loan_friend, validate_loan_book, validate_rating
)
```